In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures
import pandas as pd
from tqdm import tqdm
from scipy.optimize import minimize, basinhopping

plt.rcParams['text.latex.preamble'] = r"\usepackage{amsmath}"
plt.rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
plt.rc('text',usetex=True)

import julia
from julia import Main
from julia import Distributed
Main.eval("using QuadGK")

In [2]:
Main.eval('using Distributed')
Main.eval('addprocs(7)') 

array([2, 3, 4, 5, 6, 7, 8], dtype=int64)

In [3]:
Main.include("strong coupling\\constants.jl")
Main.include("strong coupling\\alpha_s.jl")
Main.include("anomalous dim\\cusp\\cusp.jl")
Main.include("anomalous dim\\non-cusp\\gammaH.jl")
Main.include("anomalous dim\\non-cusp\\gammaB.jl")
Main.include("anomalous dim\\non-cusp\\gammaS.jl")
Main.include("anomalous dim\\non-cusp\\gammav.jl")
Main.include("hard\\hard.jl")
Main.include("jet\\jet.jl")
Main.include("soft\\soft.jl")
Main.include("sigma.jl")
Main.include("error.jl")
Main.include("fixed order\\perturbation.jl")
Main.include("fixed order\\SCET.jl")
Main.include("fixed order\\non-singular.jl")
Main.include("fit.jl")
Main.include("NP.jl")
Main.include("renormalon.jl")

<PyCall.jlwrap renormalon_MS_func>

Range Choice

In [4]:
χ_lower = 115
χ_upper = 165

Read Data

In [5]:
#-------------------------------------------------------------------
dataset_names = ["SLD","DELPHI","TOPAZ_595","TOPAZ_533",
                 "TASSO_435","TASSO_348","MARKII","MAC"]
#-------------------------------------------------------------------

dataset_qua = ["OPAL","MAC"]  
 
df_qua = {}

for name in dataset_qua:
    file_path = f"data/{name}.csv"
    df = pd.read_csv(file_path)
    
    χ = df["CHI"]
    EEC = df["EEC"]
    ERROR = df["STAT"]
    
    df1 = pd.DataFrame({"CHI": χ, "EEC": EEC, "ERROR": ERROR})
    df_qua[name] = df1

dataset_ss = ["SLD","DELPHI","TOPAZ_595","TOPAZ_533","TASSO_435","TASSO_348","MARKII"]

df_ss = {}

for name in dataset_ss:
    file_path = f"data/{name}.csv"
    df = pd.read_csv(file_path)
    
    χ = df["CHI"]
    EEC = df["EEC"]
    ERROR = np.sqrt(df["STAT"]**2+df["SYS"]**2)
    
    df1 = pd.DataFrame({"CHI": χ, "EEC": EEC, "ERROR": ERROR})
    df_ss[name] = df1

df = df_qua
df.update(df_ss)

df_truncated = {}
χ = {}
EEC = {}
ERROR = {}

for name in dataset_names:
    df_truncated[name] = df[name][(df[name]["CHI"] <= χ_upper) & (df[name]["CHI"] >= χ_lower)]

    χ[name] = np.array(df_truncated[name]["CHI"])
    EEC[name] = np.array(df_truncated[name]["EEC"])
    ERROR[name] = np.array(df_truncated[name]["ERROR"])

Q_list = {}
Q_list["SLD"]       = 91.2
Q_list["OPAL"]      = 91.2
Q_list["DELPHI"]    = 91.2
Q_list["TOPAZ_595"] = 59.5
Q_list["TOPAZ_533"] = 53.3
Q_list["TASSO_435"] = 43.5
Q_list["TASSO_348"] = 34.8
Q_list["MARKII"]    = 29.0
Q_list["MAC"]       = 29.0

Chi-Square calculation

In [6]:
def DELTA_func(ylist,PRED):
    l = len(ylist)
    theory = np.zeros(l)
    for i in range(l):
        simpson = 1/12*(PRED[0][i] + 4*PRED[1][i] + 2*PRED[2][i] + 4*PRED[3][i] + PRED[4][i])
        theory[i] = np.nan_to_num(simpson,nan=0.0)
    return theory - ylist

def CHI2_func(DELTA,ERROR):
    chi_squared = np.sum(np.square(DELTA/ERROR))
    return chi_squared

Define Objective

In [7]:
def objective(params, df, Q_list, dataset_names, if_renormalon, ratio_list):

    #----------
    XLL="N4LL"
    XLO=2 # 1:LO 2:NLO
    #----------
    
    N=5
    chi2 = 0
    length = 0
    
    αs = params[0]
    params_noαs = params[1:]

    for name in dataset_names:

        xlist={}
        xlist_mid=np.array(df[name]["CHI"])
        binsize=abs(xlist_mid[1]-xlist_mid[0]) # Evenly sized bins assumed
        for i in range(N):
            xlist[i]=xlist_mid-binsize/2+i*binsize/4

        ylist=np.array(df[name]["EEC"])
        ERROR_list=np.array(df[name]["ERROR"])

        l = len(xlist_mid)
        length = length + l

        PRED={}

        if if_renormalon:
            Ω1 = params_noαs[0]
            parameters = params_noαs[1:]
            if len(parameters)> 0:
                for quartile in range(N):
                    PRED[quartile] = Main.model(xlist=xlist[quartile], αs=αs, Q=Q_list[name], XLL=XLL, XLO=XLO, parameters=parameters, Ω1=Ω1
                                                , μS_ratio=ratio_list["μS_ratio"], μJ_ratio=ratio_list["μJ_ratio"]
                                                , νS_ratio=ratio_list["νS_ratio"], νJ_ratio=ratio_list["νJ_ratio"]
                                                , μH_ratio=ratio_list["μH_ratio"])
                DELTA =  DELTA_func(ylist,PRED)
            else: 
                for quartile in range(N):
                    PRED[quartile] = Main.model(xlist=xlist[quartile], αs=αs, Q=Q_list[name], XLL=XLL, XLO=XLO, Ω1=Ω1
                                                , μS_ratio=ratio_list["μS_ratio"], μJ_ratio=ratio_list["μJ_ratio"]
                                                , νS_ratio=ratio_list["νS_ratio"], νJ_ratio=ratio_list["νJ_ratio"]
                                                , μH_ratio=ratio_list["μH_ratio"])             
                DELTA =  DELTA_func(ylist,PRED)               
        else:
            parameters = params_noαs
            if len(parameters) > 0:
                for quartile in range(N):
                    PRED[quartile] = Main.model(xlist=xlist[quartile], αs=αs, Q=Q_list[name], XLL=XLL, XLO=XLO, parameters=parameters
                                                , μS_ratio=ratio_list["μS_ratio"], μJ_ratio=ratio_list["μJ_ratio"]
                                                , νS_ratio=ratio_list["νS_ratio"], νJ_ratio=ratio_list["νJ_ratio"]
                                                , μH_ratio=ratio_list["μH_ratio"])                                                               
                DELTA =  DELTA_func(ylist,PRED)             
            else: 
                for quartile in range(N):
                    PRED[quartile] = Main.model(xlist=xlist[quartile], αs=αs, Q=Q_list[name], XLL=XLL, XLO=XLO
                                                , μS_ratio=ratio_list["μS_ratio"], μJ_ratio=ratio_list["μJ_ratio"]
                                                , νS_ratio=ratio_list["νS_ratio"], νJ_ratio=ratio_list["νJ_ratio"]
                                                , μH_ratio=ratio_list["μH_ratio"])                                                             
                DELTA =  DELTA_func(ylist,PRED)              

        chi_square = CHI2_func(DELTA,ERROR_list)
        chi2 = chi2 + chi_square

    return chi2/length

Define minimization related

In [8]:
import sys

class ProgressCallback:
    def __init__(self, df, Q_list, dataset_names, if_renormalon, ratio_list):
        self.df = df
        self.Q_list = Q_list
        self.dataset_names = dataset_names
        self.if_renormalon = if_renormalon
        self.ratio_list = ratio_list

    def __call__(self, params):
        chi2_bydof = objective(params, self.df, self.Q_list, self.dataset_names, self.if_renormalon, self.ratio_list)
        sys.stdout.flush()

class MyBounds:
    def __init__(self, xmax, xmin):
        self.xmax = np.array(xmax)
        self.xmin = np.array(xmin)

    def __call__(self, **kwargs):
        x = kwargs["x_new"]
        tmax = bool(np.all(x <= self.xmax))
        tmin = bool(np.all(x >= self.xmin))
        return tmax and tmin

Random Ratios

In [9]:
def custom_random():
    n = np.random.uniform(1.0, 2.0)  
    if np.random.rand() < 0.5:
        return n  
    else:
        return 1 / n  

def random_ratios(n):
    ratio_matrix = pd.DataFrame(columns=['μS_ratio', 'μJ_ratio', 'νS_ratio', 'νJ_ratio', 'μH_ratio', 'bmax_ratio'])
    
    while len(ratio_matrix) < n:
        μS_ratio, μJ_ratio, νS_ratio, νJ_ratio, μH_ratio, bmax_ratio = custom_random(), custom_random(), custom_random(), custom_random(), custom_random(), custom_random()
        
        if (abs(μS_ratio / μJ_ratio) <= 2 and abs(μS_ratio / νS_ratio) <= 2 and abs(νJ_ratio / νS_ratio) <= 2 and abs(μH_ratio / μJ_ratio) <= 2 and
                abs(μS_ratio / μJ_ratio) >= 0.5 and abs(μS_ratio / νS_ratio) >= 0.5 and abs(νJ_ratio / νS_ratio) >= 0.5 and abs(μH_ratio / μJ_ratio) >= 0.5 and
                bmax_ratio >= 2/3 and bmax_ratio <= 3/2):

            ratio_matrix.loc[len(ratio_matrix)] = [μS_ratio, μJ_ratio, νS_ratio, νJ_ratio, μH_ratio, bmax_ratio]
    
    return ratio_matrix

Test

In [11]:
objective([0.1181, 3.53, -0.665, -0.173], df_truncated, Q_list, dataset_names, False, pd.Series({'μS_ratio':1.0,'μJ_ratio':1.0,'νS_ratio':1.0,'νJ_ratio':1.0,'μH_ratio':1.0,'bmax_ratio':1.0}))

1.560892232121887

Minimization

In [12]:
ratios = random_ratios(50)
n = len(ratios)

In [13]:
print(ratios)

    μS_ratio  μJ_ratio  νS_ratio  νJ_ratio  μH_ratio  bmax_ratio
0   0.778723  1.115739  0.915135  0.955739  1.859244    1.323450
1   1.733384  1.735374  1.686042  1.742323  0.894119    1.470736
2   0.719716  0.779431  0.633042  0.720778  0.629777    1.106542
3   1.179771  1.377043  1.563093  1.733861  0.750698    1.466093
4   0.603435  1.035275  0.771913  0.787839  0.677590    0.769538
5   0.737342  0.500433  0.846050  1.517155  0.556736    1.315081
6   0.683233  0.940061  0.524809  0.753524  1.660416    0.735589
7   1.291455  1.110462  1.401422  1.549950  1.477054    1.017737
8   1.202718  1.930519  1.144745  1.339313  1.008872    0.835485
9   0.996890  1.943971  1.443494  0.883369  1.218660    1.081403
10  0.631269  1.173816  0.931218  1.162797  1.698953    1.215695
11  1.043882  1.900747  1.658831  1.665866  1.284526    0.701793
12  1.358225  0.865358  1.225286  1.250441  0.914327    1.250620
13  1.140205  1.052174  0.770644  0.954593  1.403877    1.200606
14  1.636449  1.473280  1

In [14]:
if_renormalon = False

In [15]:
results_df = pd.DataFrame(columns=['αs', 'params','chi2/dof', 'ratios_list'])

initial_params = [0.1181, 3.53, -0.665, -0.173] #best_params

bounds = [ [0.11 , 0.13], # αs
           #[-1.0 , 1.0  ], # Ω1
            [0.0, 6.0  ], # a1
            [-3.0, 3.0  ], # a2
            [-3.0, 3.0  ], # a3         
        ]

for i in tqdm(range(n)):

    ith_row = ratios.iloc[i]
    μS_ratio = ith_row['μS_ratio']
    μJ_ratio = ith_row['μJ_ratio']
    νS_ratio = ith_row['νS_ratio']
    νJ_ratio = ith_row['νJ_ratio']
    μH_ratio = ith_row['μH_ratio']
    bmax_ratio = ith_row['bmax_ratio']

    ratio_list={}
    ratio_list["μS_ratio"]=μS_ratio
    ratio_list["μJ_ratio"]=μJ_ratio
    ratio_list["νS_ratio"]=νS_ratio
    ratio_list["νJ_ratio"]=νJ_ratio
    ratio_list["μH_ratio"]=μH_ratio
    ratio_list["bmax_ratio"]=bmax_ratio    

    bounds_T = [list(t) for t in zip(*bounds)]

    progress_callback = ProgressCallback(df_truncated, Q_list, dataset_names, if_renormalon, ratio_list)

    minimizer_kwargs = {
        "method": "L-BFGS-B",
        "bounds": bounds,
        "args": (df_truncated, Q_list, dataset_names, if_renormalon, ratio_list),
        "options": {'maxiter': 100, 'disp': True, "ftol": 10**(-5)},
        "callback": progress_callback
    }

    result = basinhopping(
        objective,
        initial_params,
        minimizer_kwargs=minimizer_kwargs,
        niter=0,
        accept_test=MyBounds(xmax=bounds_T[1],xmin=bounds_T[0])
    )

#-----------------------------------------------------------------------------

    optimal = np.round(result.x,5)
    chi2_per_dof = result.fun

    optimal_params = optimal[1:]

    new_row = pd.Series({
        'αs': optimal[0], 
        'params': optimal_params, 
        'chi2/dof': result.fun,
        'ratios_list': ratio_list 
    })
    results_df = pd.concat([results_df, new_row.to_frame().T], ignore_index=True)

display(results_df)

100%|██████████| 50/50 [51:20<00:00, 61.61s/it]


,αs,params,chi2/dof,ratios_list
0,0.11836,"[3.4677, -0.42845, -0.25661]",1.653206,"{'μS_ratio': 0.7787234701359167, 'μJ_ratio': 1..."
1,0.11711,"[3.75314, -1.06857, -0.05567]",1.489848,"{'μS_ratio': 1.7333838613547456, 'μJ_ratio': 1..."
2,0.11846,"[3.40335, -0.2945, -0.29989]",1.676629,"{'μS_ratio': 0.7197164638937481, 'μJ_ratio': 0..."
3,0.11746,"[3.59486, -0.87448, -0.11585]",1.534436,"{'μS_ratio': 1.1797705043468696, 'μJ_ratio': 1..."
4,0.11816,"[3.26692, -0.14138, -0.36701]",1.797813,"{'μS_ratio': 0.603434754572047, 'μJ_ratio': 1...."
5,0.11932,"[3.38565, -0.21245, -0.29529]",1.611741,"{'μS_ratio': 0.7373415814471664, 'μJ_ratio': 0..."
6,0.1187,"[3.35251, -0.20029, -0.31801]",1.716662,"{'μS_ratio': 0.6832333681763323, 'μJ_ratio': 0..."
7,0.1177,"[3.59837, -0.87444, -0.09788]",1.51735,"{'μS_ratio': 1.2914550597881402, 'μJ_ratio': 1..."
8,0.11767,"[3.63604, -0.91508, -0.10503]",1.530209,"{'μS_ratio': 1.2027175829477232, 'μJ_ratio': 1..."
9,0.11838,"[3.53069, -0.6739, -0.18459]",1.569226,"{'μS_ratio': 0.9968896023700975, 'μJ_ratio': 1..."


Store Result to a File

In [16]:
results_df.to_csv("result//fit_N4LL_3.csv", index=False)

In [ ]:
Main.eval('rmprocs(workers())')

<PyCall.jlwrap Task (runnable) @0x000001ebac4989c0>